In [5]:
# Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [6]:
train = pd.read_csv(
    "22121477/train_lending_club.csv"
)

test = pd.read_csv(
    "22121477/test_lending_club.csv"
)

In [7]:
features = [
    "fico_range_low",
    "dti",
    "annual_inc",
    "revol_util",
    "total_acc",
    "open_acc",
    "pub_rec",
    "pub_rec_bankruptcies",
    "mort_acc",
    "emp_length"
]

target = "loan_status"

loan_status_labeled = test["loan_status"].notna()

X_train = train[features]
y_train = train[target]
X_test = test.loc[loan_status_labeled, features]
y_test = test.loc[loan_status_labeled, target]

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)
# validate data
print(X_train.dtypes)
print(X_train.isna().sum())
print(X_test.isna().sum())

(236846, 10)
(236846,)
(93607, 10)
(93607,)
fico_range_low          float64
dti                     float64
annual_inc              float64
revol_util              float64
total_acc               float64
open_acc                float64
pub_rec                 float64
pub_rec_bankruptcies    float64
mort_acc                float64
emp_length              float64
dtype: object
fico_range_low          0
dti                     0
annual_inc              0
revol_util              0
total_acc               0
open_acc                0
pub_rec                 0
pub_rec_bankruptcies    0
mort_acc                0
emp_length              0
dtype: int64
fico_range_low          0
dti                     0
annual_inc              0
revol_util              0
total_acc               0
open_acc                0
pub_rec                 0
pub_rec_bankruptcies    0
mort_acc                0
emp_length              0
dtype: int64


In [8]:
model_pipeline = Pipeline([
    (
        "preprocessing_step",
        StandardScaler()
    ),
    (
        "model_step",
        LogisticRegression(
            max_iter= 1000
        )
    )
])

model_pipeline.fit(
    X_train,
    y_train
)

predicted_classes = model_pipeline.predict(
    X_test
)

predicted_probabilities = model_pipeline.predict_proba(
    X_test
)

model_pipeline.classes_

array([0, 1])

In [9]:
print(np.shape(predicted_classes))

print(np.shape(predicted_probabilities))

print(predicted_probabilities[:5])





(93607,)
(93607, 2)
[[0.00748773 0.99251227]
 [0.38710138 0.61289862]
 [0.18721613 0.81278387]
 [0.08484405 0.91515595]
 [0.18245777 0.81754223]]


In [10]:
print(f"Baseline model predicted success(1) vs defaults (0): \n{pd.Series(predicted_classes).value_counts()}\n")
print(f"Actual success(1) vs defaults (0): \n{pd.Series(y_test).value_counts()}")

Baseline model predicted success(1) vs defaults (0): 
1    93149
0      458
Name: count, dtype: int64

Actual success(1) vs defaults (0): 
loan_status
1.0    78859
0.0    14748
Name: count, dtype: int64


In [11]:
cm = confusion_matrix(
    y_test,
    predicted_classes,
    labels=[0, 1]
)

cm_df = pd.DataFrame(
    cm,
    index=["Actual default (0)", "Actual success (1)"],
    columns=["Predicted default (0)", "Predicted success (1)"]
)

cm_df

,Predicted default (0),Predicted success (1)
Actual default (0),140,14608
Actual success (1),318,78541


In [12]:
accuracy = accuracy_score(
    y_test,
    predicted_classes
)

default_precision = precision_score(
    y_test,
    predicted_classes,
    pos_label=0
)

default_recall = recall_score(
    y_test,
    predicted_classes,
    pos_label=0
)

default_f1 = f1_score(
    y_test,
    predicted_classes,
    pos_label=0
)

print("Accuracy:", accuracy)
print("Default precision:", default_precision)
print("Default recall:", default_recall)
print("Default F1-score:", default_f1)

Accuracy: 0.8405461130043693
Default precision: 0.3056768558951965
Default recall: 0.009492812584757255
Default F1-score: 0.018413784032618702


In [13]:
default_probabilities = predicted_probabilities[:,0]

average_probability_for_defaults = default_probabilities[y_test == 0].mean()

average_probability_for_successes = default_probabilities[y_test == 1].mean()

print(f"Average predicted default probability for actual defaults: {100 * average_probability_for_defaults:.2f}%")

print(f"Average predicted default probability for actual successes: {100 * average_probability_for_successes:.2f}%")

Average predicted default probability for actual defaults: 21.01%
Average predicted default probability for actual successes: 15.15%


In [14]:
print("Actual defaults:")
print(
    np.percentile(
        default_probabilities[y_test == 0],
        [25, 50, 75, 90, 95]
    )
)

print("\nActual successes:")
print(
    np.percentile(
        default_probabilities[y_test == 1],
        [25, 50, 75, 90, 95]
    )
)

Actual defaults:
[0.14754081 0.20115369 0.26077964 0.32117259 0.35857547]

Actual successes:
[0.09353338 0.14217615 0.19576727 0.25128498 0.2890208 ]


In [15]:
thresholds = [0.10, 0.15, 0.20, 0.24, 0.25, 0.30, 0.35, 0.40, 0.50]

for threshold in thresholds:

    threshold_predictions = (
        default_probabilities >= threshold
    ).astype(int)

    threshold_predictions = 1 - threshold_predictions

    threshold_default_recall = recall_score(
        y_test,
        threshold_predictions,
        pos_label=0
    )

    threshold_default_precision = precision_score(
        y_test,
        threshold_predictions,
        pos_label=0
    )

    print(
        f"Threshold: {threshold:.2f} | "
        f"Default recall (Defaulters Caught): {threshold_default_recall:.3f} | "
        f"Default precision (Prediction Accuracy): {threshold_default_precision:.3f}"
    )

Threshold: 0.10 | Default recall (Defaulters Caught): 0.908 | Default precision (Prediction Accuracy): 0.191
Threshold: 0.15 | Default recall (Defaulters Caught): 0.738 | Default precision (Prediction Accuracy): 0.231
Threshold: 0.20 | Default recall (Defaulters Caught): 0.506 | Default precision (Prediction Accuracy): 0.288
Threshold: 0.24 | Default recall (Defaulters Caught): 0.324 | Default precision (Prediction Accuracy): 0.332
Threshold: 0.25 | Default recall (Defaulters Caught): 0.287 | Default precision (Prediction Accuracy): 0.344
Threshold: 0.30 | Default recall (Defaulters Caught): 0.144 | Default precision (Prediction Accuracy): 0.397
Threshold: 0.35 | Default recall (Defaulters Caught): 0.059 | Default precision (Prediction Accuracy): 0.408
Threshold: 0.40 | Default recall (Defaulters Caught): 0.023 | Default precision (Prediction Accuracy): 0.354
Threshold: 0.50 | Default recall (Defaulters Caught): 0.009 | Default precision (Prediction Accuracy): 0.306


In [16]:
for threshold in thresholds:

    threshold_predictions = (
        default_probabilities >= threshold
    ).astype(int)

    threshold_predictions = 1 - threshold_predictions

    predicted_default_count = np.sum(
        threshold_predictions == 0
    )

    predicted_success_count = np.sum(
        threshold_predictions == 1
    )

    print(
        f"Threshold: {threshold:.2f} | "
        f"Predicted defaults: {predicted_default_count} | "
        f"Predicted successes: {predicted_success_count}"
    )

Threshold: 0.10 | Predicted defaults: 70077 | Predicted successes: 23530
Threshold: 0.15 | Predicted defaults: 47185 | Predicted successes: 46422
Threshold: 0.20 | Predicted defaults: 25955 | Predicted successes: 67652
Threshold: 0.24 | Predicted defaults: 14412 | Predicted successes: 79195
Threshold: 0.25 | Predicted defaults: 12317 | Predicted successes: 81290
Threshold: 0.30 | Predicted defaults: 5350 | Predicted successes: 88257
Threshold: 0.35 | Predicted defaults: 2122 | Predicted successes: 91485
Threshold: 0.40 | Predicted defaults: 947 | Predicted successes: 92660
Threshold: 0.50 | Predicted defaults: 458 | Predicted successes: 93149


In [17]:
from sklearn.metrics import roc_auc_score, average_precision_score

default_target = (y_test == 0).astype(int)

roc_auc = roc_auc_score(
    default_target,
    default_probabilities
)

pr_auc = average_precision_score(
    default_target,
    default_probabilities
)

print("ROC-AUC:", roc_auc)
print("PR-AUC:", pr_auc)

ROC-AUC: 0.697250728765148
PR-AUC: 0.2897398413819147


# Baseline Model Conclusion

The baseline logistic regression model used borrower information including FICO score, debt-to-income ratio, annual income, revolving credit utilization, account history, public records, mortgage accounts, and employment length to estimate each loan’s probability of default. The interest rate was excluded from this model so that it could later be compared with an expanded model containing additional information.

The baseline model achieved a ROC-AUC of approximately **0.70**. This means that the model had some ability to rank riskier loans above safer loans, but its performance was only moderate. Its PR-AUC was approximately **0.29**, which was better than random performance but showed that the model struggled to identify defaults accurately.

At the standard 0.50 classification threshold, the model achieved:

- Accuracy: **84.1%**
- Default precision: **30.6%**
- Default recall: **0.9%**
- Default F1-score: **1.8%**

Although the accuracy appeared relatively high, accuracy was not a very useful measure in this case because most loans in the test set were successful. The model predicted only **458 defaults**, while there were actually **14,748 defaults**. Therefore, the model classified nearly every loan as successful and failed to identify most of the loans that defaulted.

The confusion matrix showed this clearly:

| Actual Outcome | Predicted Default | Predicted Success |
|---|---:|---:|
| Actual Default | 140 | 14,608 |
| Actual Success | 318 | 78,541 |

The model correctly identified only 140 of the 14,748 actual defaults. This produced a default recall of approximately 0.9%. In practical terms, the model missed more than 99% of the loans that eventually defaulted. Its default precision was approximately 30.6%, meaning that only about three out of every ten loans labeled as defaults actually defaulted.

The predicted probabilities provided slightly better separation than the final class predictions. The median predicted default probability was approximately **20.1%** for actual defaults and **14.2%** for successful loans. This shows that actual defaults generally received somewhat higher risk scores than successful loans, but the difference was not large enough for the default and successful groups to be separated reliably at the standard 0.50 threshold.

Overall, the baseline model provided a useful starting point but was not strong enough to support a lending or loan-purchase decision on its own. Its main weakness was that it treated nearly all loans as successful because defaults were less common than successful repayments. As a result, its high accuracy was misleading, while its very low default recall showed that it was poor at finding risky loans.

The baseline model establishes a comparison point for the expanded model. After additional information—particularly the interest rate—was added, the expanded model performed substantially better. Therefore, the baseline results suggest that the original borrower characteristics alone did not contain enough information to identify defaults effectively.

The baseline model is useful for this project because it shows how much predictive value is gained by adding additional origination-time information. However, it does not yet identify invisible-prime borrowers. It only provides the initial risk-ranking model against which the expanded model can be evaluated.